# Python Commands Reference

Quick reference for Python, NumPy, and SymPy commands used in this project.  
One entry per command — what it does and a minimal example. No theory.

---

## Contents

1. [Converting a SymPy Matrix to a NumPy Array](#sympy-to-numpy)
2. [Flattening a Matrix to a 1D Vector](#flatten)
3. [Hashing a File with hashlib](#hashlib)
4. [Saving and Loading Python Objects with pickle](#pickle)
5. [Working with File Paths using pathlib](#pathlib)
6. [List Comprehensions](#list-comprehension)
7. [Error Handling with raise and ValueError](#error-handling)
8. [Closures](#closure)

---

<a id='sympy-to-numpy'></a>
## 1. Converting a SymPy Matrix to a NumPy Array

**Goal:** evaluate all symbolic expressions to floats so NumPy tools (like `eigvalsh`) can work on the result.

```python
import numpy as np

# M is a SymPy Matrix with symbolic entries
M_num = np.array(M.evalf(), dtype=float)
```

- `.evalf()` — evaluates every element to a floating-point number
- `np.array(..., dtype=float)` — converts the result into a NumPy array

Use this any time you need to go from symbolic → numerical for testing or simulation.

---

<a id='flatten'></a>
## 2. Flattening a Matrix to a 1D Vector

**Goal:** convert a 2D array into a single 1D array so it can be concatenated with other vectors or passed to functions like `np.linalg.norm`.

```python
import numpy as np

R = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

R.flatten()  # returns array([1, 2, 3, 4, 5, 6, 7, 8, 9])
```

Reads left-to-right, row by row. Common use in IK — combining a rotation error matrix with a position error vector into one flat error signal:

```python
error = np.concatenate([pos_error, rot_error.flatten()])
```

---

<a id='hashlib'></a>
## 3. Hashing a File with hashlib

**Goal:** produce a short fixed-length string that uniquely identifies the contents of a file. If the file changes, the hash changes.

```python
import hashlib

with open("configs/robots/example_6dof.yaml", "rb") as f:
    file_hash = hashlib.md5(f.read()).hexdigest()

print(file_hash)  # e.g. "a3f1c9d82b..."
```

- `"rb"` — open in **binary** read mode (required for hashing — text mode can alter line endings)
- `hashlib.md5(...)` — creates an MD5 hash object from the bytes
- `.hexdigest()` — returns the hash as a readable hex string

Use this to build a cache key: if the YAML changes, the hash changes and the old cache file is ignored.

**Reference:** https://docs.python.org/3/library/hashlib.html

---

<a id='pickle'></a>
## 4. Saving and Loading Python Objects with pickle

**Goal:** serialise a Python object (like a tuple of SymPy expressions) to a file on disk, then load it back in a later session.

```python
import pickle

# Saving — open in binary write mode
with open("cache/my_robot.pkl", "wb") as f:
    pickle.dump((M, C, g, theta_syms, theta_dot_syms), f)

# Loading — open in binary read mode
with open("cache/my_robot.pkl", "rb") as f:
    M, C, g, theta_syms, theta_dot_syms = pickle.load(f)
```

- `"wb"` / `"rb"` — binary write / binary read (pickle files are binary, not text)
- `pickle.dump(obj, f)` — writes the object into the open file
- `pickle.load(f)` — reads and reconstructs the object from the file
- The tuple is unpacked directly on load — same as any Python tuple assignment

SymPy expressions are picklable, so `M`, `C`, `g`, and symbol lists can all be stored this way.

**Reference:** https://docs.python.org/3/library/pickle.html

---

<a id='pathlib'></a>
## 5. Working with File Paths using pathlib

**Goal:** build file paths, check if files exist, and create directories — without manually joining strings.

```python
from pathlib import Path

# Build a path
cache_dir = Path("cache")
cache_file = cache_dir / "example_6dof_a3f1c9d82b.pkl"

# Check if a file exists
if cache_file.exists():
    print("cache hit")

# Create a directory (and any missing parents) — does nothing if it already exists
cache_dir.mkdir(parents=True, exist_ok=True)
```

- `Path("cache") / "filename.pkl"` — the `/` operator joins path segments (works on all operating systems)
- `.exists()` — returns `True` if the file or directory is present on disk
- `.mkdir(parents=True, exist_ok=True)` — creates the directory; `exist_ok=True` prevents an error if it already exists

**Reference:** https://docs.python.org/3/library/pathlib.html

---

<a id='list-comprehension'></a>
## 6. List Comprehensions

**Goal:** build a list by applying an expression to each item in a sequence — a concise alternative to a `for` loop with `.append()`.

```python
# Loop version
results = []
for i in range(5):
    results.append(i * 2)

# List comprehension — same result, one line
results = [i * 2 for i in range(5)]
# [0, 2, 4, 6, 8]
```

The pattern is always: `[expression for item in iterable]`

Real example from `path_generator.py` — sample a path at n evenly spaced time values:

```python
t_values = np.linspace(0, 1, n_steps)
poses = [path_fn(t) for t in t_values]
```

You can also add a condition to filter items:

```python
evens = [x for x in range(10) if x % 2 == 0]
# [0, 2, 4, 6, 8]
```

**Reference:** https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions

---

<a id='error-handling'></a>
## 7. Error Handling with raise and ValueError

**Goal:** catch bad inputs early and give the caller a clear message, rather than letting the program crash with a confusing error later.

The most common pattern is to check a condition at the top of a function and `raise` an exception if it fails:

```python
def make_spline_path(waypoints):
    if len(waypoints) < 2:
        raise ValueError("make_spline_path requires at least 2 waypoints, got 1.")
    # ... rest of function
```

- `raise` — immediately stops the function and signals that something went wrong
- `ValueError` — the built-in exception type for "you gave me a bad value"; use it for invalid inputs
- The string is the error message the caller will see

Other common exception types:

| Exception | When to use |
|---|---|
| `ValueError` | Bad value for a valid type (e.g. negative radius, too few waypoints) |
| `TypeError` | Wrong type entirely (e.g. passing a string where a list is expected) |
| `NotImplementedError` | Function exists but isn't implemented yet (e.g. sensor path stub) |
| `FileNotFoundError` | A file or path doesn't exist |

You can also *catch* exceptions with `try / except` if you want to handle the error rather than crash:

```python
try:
    result = make_spline_path(waypoints)
except ValueError as e:
    print(f"Path error: {e}")
```

**Reference:** https://docs.python.org/3/tutorial/errors.html

---

<a id='closure'></a>
## 8. Closures

**What it is:** a function defined *inside* another function that "remembers" variables from the outer scope — even after the outer function has finished running.

The outer function takes configuration as arguments. The inner function (often called `f`) is what gets returned, and it silently carries those arguments with it.

```python
def make_multiplier(factor):
    # 'factor' lives in the outer scope

    def f(x):
        # f can still see 'factor' even after make_multiplier() has returned
        return x * factor

    return f

double = make_multiplier(2)
print(double(5))   # 10
print(double(9))   # 18
```

`factor` is "closed over" by `f` — that's why the pattern is called a closure.

---

**Why it matters here:** every path generator in `path_generator.py` uses this pattern. The outer function (`make_line_path`, `make_spline_path`, etc.) accepts the configuration parameters for a specific path. It returns an inner function `f(t)` that the rest of the simulator calls to evaluate the path at any time `t ∈ [0, 1]`.

```python
def make_line_path(start, end):
    # 'start' and 'end' are captured here, in the outer scope

    def f(t):
        # f(t) can use 'start' and 'end' even though make_line_path has already returned
        return (1 - t) * start + t * end

    return f

path = make_line_path(np.array([0, 0, 0]), np.array([1, 1, 1]))
pose_at_half = path(0.5)   # array([0.5, 0.5, 0.5])
```

The simulator only ever calls `path(t)` — it never needs to know about `start` or `end` directly. The closure keeps them safe inside `f`.

**Reference:** https://docs.python.org/3/faq/programming.html#why-do-lambdas-defined-in-a-loop-with-different-values-all-return-the-same-result

---